# **AQM SM 2021**

## **V. Testing for heteroskedasticity, multicollinearity, serial autocorrelation**

**To do:**
- Download monthly data for at least 10 asset prices and at least 10 macroeconomic variables, across countries
- For each macroeconomic variable, create a loop where you:
    * Run multivariate OLS regressions using all combinations of 1 and 10 explanatory macroeconomic variables
    * Save the main regression results to the table
    * For each regression, test for autocorrelation, heteroskedasticity and multicollinearity, save the results into the table
- The main output table will include the key statistics from each regression as well as the results of the tests
- Figure out which models have the highest explanatory power and are most promising

## **Imports**

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import xlsxwriter
from scipy import stats
from datetime import datetime
from pandas.plotting import register_matplotlib_converters
import sqlite3
from sqlite3 import Error
from sklearn import linear_model
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats.stattools import durbin_watson
import statsmodels.stats.api as sms
from statsmodels.compat import lzip
import os
import sys
import itertools
from itertools import combinations


## **Creating a Data Base**

In [ ]:
def DataBaseConnexion(db_file):
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        print(sqlite3.sqlite_version)
    except Error as e:
        print(e)
    finally:
        if conn:
            conn.close()

os.makedirs('output', exist_ok=True)

if __name__ == '__main__':
    DataBaseConnexion('output/AQM.db')

conn = sqlite3.connect('output/AQM.db')
c = conn.cursor()
print('Connexion done')
conn = sqlite3.connect('output/AQM.db')
print("Opened database successfully")
conn = sqlite3.connect('output/AQM.db')
c = conn.cursor()


## **Run the OLS regression**

In [ ]:
def ev_dw(b):
    if b==2:
        print('Durbin Watson test:',b, '. There is no autocorelation detected in the sample')
    elif 0<b<2:
        print('Durbin Watson test:',b, '. There is positive autocorelation detected in the sample')
    else:
        print('Durbin Watson test:',b, '. There is negative autocorelation detected in the sample')


def ev_BP(x):
    if x[1]>0.05:
        print('Breush Pagan test:',x,'. H0 rejected, Heteroscedasticity exists')
    else:
        print('Breush Pagan test:',x,'. H0 accepted, No Heteroscedasticity')

def ev_JB(x):
    if x[2]<(-1) or x[2]>1:
        print('Jarque Bera test:',x,'. The distribution is highly skewed')
    elif (-1)<x[2]<(-0.5) or (0.5)<x[2]<1:
        print('Jarque Bera test:',x,'. The distribution is moderately skewed')
    else:
        print('Jarque Bera test:',x,'. The distribution is aproximatively symetric')

7 Variables OLS Regressions

In [ ]:
d5 = pd.read_excel('data/AQM_1.xlsx', sheet_name='StockPrice',index_col=0)
d6 = pd.read_excel('data/AQM_1.xlsx', sheet_name='MacroIndicators', index_col=0)


In [ ]:
import aqm_lib
from concurrent.futures import ProcessPoolExecutor

Companies = ['BP',                          #Define a list with the companies we selected for our analysis
             'Total SE',
             'Volkswagen AG',
             'Bayerische Motoren Werke AG',
             'Allianz SE',
             'Deutsche Bank AG',
             'L Oreal SA',
             'Schneider Electric SE',
             'Danone SA',
             'Vinci SA']
                                            #Define a new list with the variables
MacroInd = ['EU Consummer Price All Items','EU Construction Output','EU Unemployment Rate','EU Consumer Price Education','EU Fixed Interest Rate','EU Economic Sentiment Indicator','EU Total Production Construction Excluded','Domestic Producer Prices','EU Retail Sales Turnover','EU Exports']

cc = list(combinations(MacroInd,7))         #Define a list with all the combination possible of 7 variables out of 10 which is (10!)/(3!7!)= 120 combinations of variables possible

os.makedirs('output/BestModels', exist_ok=True)
os.makedirs('output/7VarOLS_PredictedData', exist_ok=True)
os.makedirs('output/ObservedVsPredictedPricesGraphs', exist_ok=True)
os.makedirs('output/ResidualsGraphs', exist_ok=True)

#Run each company's full 120-combination loop in its own worker process (10 tasks,
#not 1,200 - a single OLS fit is only a few milliseconds, so per-task process-pool
#overhead would dominate at combination-level granularity; see the feature 006
#research notes). Each worker writes only its own company-namespaced files (its
#7VarOLS_PredictedData workbook, its ObsVsPred/Residuals/BestModels PNGs), so there
#is no cross-worker race - different companies never share a filename.
company_work_items = [(company, cc) for company in Companies]

with ProcessPoolExecutor(initializer=aqm_lib.init_worker, initargs=('data/AQM_1.xlsx',)) as executor:
    company_results = list(executor.map(aqm_lib.fit_company, company_work_items))

results_by_company = {r.company: r for r in company_results}

w2 = pd.ExcelWriter('output/BestModels/AQM_BestResults.xlsx',engine='xlsxwriter') #create a new excel file for each company were we are going to save the predictions. WE get 10 excel files with 120 sheets each

for i in Companies:                        #Define a first lookp that is going to calculate the stoch price fo each company

    best = results_by_company[i]
    dm = str(best.best_combo_index - 1)

    print('Multivariate OLS regressions for '+i+' succesful run')
    print('Maximum adjusted R^2 is: ', best.max_adj_r2, '. Minimum adjusted R^2 is: ', best.min_adj_r2)
    print('The best 7 multivariate OLS regression for ', i, ' with 7 variables is the model that includes the variables: ', best.best_combination)
    print('     ')
    print(best.best_summary_text)

    #We reuse the winning combination's already-computed result -- no second OLS
    #fit, and its plots were already written by the worker (fit_company) into
    #output/BestModels/ and output/ObservedVsPredictedPricesGraphs|ResidualsGraphs/.
    #We run the regression for the best model and predict the stock price, later we save it in an excel file called AQM_BestResults
    regression11 = pd.DataFrame(d5[i])      #The observed stock price
    regression11['Prediction'] = pd.Series(best.best_predictions, index=best.best_index_labels)   #Calculate the predicted values
    regression11.to_excel(w2, sheet_name= i) #Save in an excel file the Observed and predicted values
    regression11.to_sql(name='Best7VarOLS_'+i, con = conn)

    #Jarque Bera test for each best OLS 7 variable models
    ev_JB(best.best_jb_result)

    #Test Breusch Pagan for each best 7variable OLS regression
    ev_BP(best.best_bp_result)

    ev_dw(best.best_dw_statistic)

    print('     ')
w2.close()

print('7 Variable OLS regression analysis complete')
print('Results saved to AQM data base')




```
 10 Variables OLS Regressions
```



In [ ]:
Companies2 = ['BP',                          #Define a list with the companies we selected for our analysis
             'Total SE',
             'Volkswagen AG',
             'Bayerische Motoren Werke AG',
             'Allianz SE',
             'Deutsche Bank AG',
             'L Oreal SA',
             'Schneider Electric SE',
             'Danone SA',
             'Vinci SA']
                                            #Define a new list with the variables
MacroInd2 = ['EU Consummer Price All Items',
            'EU Construction Output',
            'EU Unemployment Rate',
            'EU Consumer Price Education',
            'EU Fixed Interest Rate',
            'EU Economic Sentiment Indicator',
            'EU Total Production Construction Excluded',
            'Domestic Producer Prices',
            'EU Retail Sales Turnover',
            'EU Exports']

p = 0

os.makedirs('output/10VarOLS', exist_ok=True)

w3 = pd.ExcelWriter('output/10VarOLS/AQM_10VarOLS.xlsx',engine='xlsxwriter') #create a new excel file for each company were we are going to save the predictions. WE get 10 excel files with 120 sheets each

for u in Companies2:                        #Define a first lookp that is going to calculate the stoch price fo each company

    p +=1
    r = str(p)                             #We transform the Company's position in the list Companies so we can later name the excel files and images accordingly with the position of the company for ex: 1 for BP and so on

    print('10 Variable OLS regression for the '+u)
    x2 = d6[MacroInd2]                          #We transform the list with the variables in the combination j in DataFrame so we can use it to define the a DataFrame for the selected combination. needed for the regression
    y2 = d5[u]                          #We define the y data frame (Observed stock prices values for each company separately according with the loop)
    x2 = sm.add_constant(x2)
    model2 = sm.OLS(y2,x2, missing = 'drop')
    results2 = model2.fit()             #Run the OLS regression
    #print(results2.summary())          #Print the OLS regression result
    adj_r2 = results2.rsquared_adj
    print('R^2 adjusted for '+u+' is: ',results2.rsquared_adj) #Print the Adjusted r^2

    regression2= pd.DataFrame(y2)      #The observed stock price
    prediction2 = results2.predict(x2)   #Calculate the predicted values
    regression2['Prediction']= prediction2
    regression2.to_excel(w3, sheet_name= r) #Save in an excel file the Observed and predicted values

    plt.plot(regression2)             #Plot the Observed and the predicted values
    labels = ['Observed','Prediction'] #Define the axis of the plot
    plt.title('Observed vs. Predicted Stock Price'+u) #Name the plot
    plt.savefig('output/10VarOLS/ObsVsPred '+u+'.png')  #Save the plot as image named with position of the company in the list and the position of the variables combination in the list xcom
    plt.close()

    #Heteroscedasticity test for each OLS regression
    residuals2 = y2 - prediction2 #Calcul of the error (residuals) for each model
    plt.plot(residuals2) #Create a plot for the residuals for each model
    plt.title('Residuals'+u) #Name the plots with the first indice that shows n - the position of the company in the list: Companies and m - position of the model combination in the combination list: xcom
    plt.savefig('output/10VarOLS/Resid '+u+'.png') #Save plots separately
    plt.close() #Close code so we can generate a new image for the next model without combining the plots

    #Normality of the residuals
    xs_with_constant2 = sm.add_constant(x2)
    name2 = ['Jarque-Bera', 'JB P-value', 'JB skewnes', 'JB kurtosis']
    test12 = sms.jarque_bera(residuals2) #Calcul or the Jarque Bera test fo the normality test
    ev_JB(test12)

    # Breush Pagan Test for Heteroskedastic residuals
    name2 = ['Lagrange multiplier statistic', 'p-value',
                'f-value', 'f p-value']
    test32 = sms.het_breuschpagan(residuals2, results2.model.exog)
    ev_BP(test32)

    #Autocorelation test
    # JB Test for Normal Distribution of Residuals
    dw_pvalue2 = sm.stats.stattools.durbin_watson(residuals2)
    ev_dw(dw_pvalue2)

    print('10 Variable OLS regressions for '+u+' succesful run')
    plt.rc('figure', figsize=(12, 7))
    plt.text(0.01, 0.05, str(results2.summary()), {'fontsize': 10}, fontproperties = 'monospace') # approach improved by OP -> monospace!
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('output/10VarOLS/Table_'+u+'.png')
    plt.close()
    print(results2.summary())
    print('     ')

w3.close()

## **Autocorrelation**

In [ ]:
Autocorrelation = sns.pairplot(d6[['EU Consummer Price All Items',
                  'EU Construction Output',
                  'EU Unemployment Rate',
                  'EU Consumer Price Education',
                  'EU Fixed Interest Rate',
                  'EU Total Production Construction Excluded',
                  'Domestic Producer Prices',
                  'EU Retail Sales Turnover',
                  'EU Exports']])
os.makedirs('output', exist_ok=True)
Autocorrelation.savefig('output/AQM_AutoCorTest.png')

## **Detecting Variable Multicolinearity**

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
x = ['EU Consummer Price All Items',
     'EU Construction Output',
     'EU Unemployment Rate',
     'EU Consumer Price Education',
     'EU Fixed Interest Rate',
     'EU Economic Sentiment Indicator',
     'EU Total Production Construction Excluded',
     'Domestic Producer Prices',
     'EU Retail Sales Turnover',
     'EU Exports']

def calc_vif(x):
    vif = pd.DataFrame()
    vif['variables'] = x.columns
    vif['VIF']=[variance_inflation_factor(x.values, i) for i in range(len(x.columns))]
    return(vif)

In [ ]:
x = d5
e = calc_vif(x)
os.makedirs('output', exist_ok=True)
e.to_excel('output/AQM_VIF.xlsx')

## **Fixing multicolinearity: Dropping values**

To fix the multicolinearity in our case, we can drop the variables:

## **Fixing multicolinearity: Combining values**

In [ ]:
d8=d6.copy()
d8['EU Consummer Price All Items']= d8.apply(lambda x: x['EU Unemployment Rate']+ x['EU Consumer Price Education']+ x['EU Economic Sentiment Indicator']+x['EU Exports'],  axis=1)
x = d8.drop(['EU Unemployment Rate','EU Consumer Price Education','EU Economic Sentiment Indicator','EU Exports'], axis= 1)
vif = calc_vif(x)
print(vif)

**Result**: low multicolinearity. 6 variables came up with satysfying values and we can proceed further to build a regression model.

## **Correlation Matrix**

In [ ]:
CorrelationMatrix = plt.figure(figsize =(10,8))
sns.heatmap(round(d6.corr(),2),annot= True)
plt.title('Correlation of variables')
plt.show()
os.makedirs('output', exist_ok=True)
CorrelationMatrix.savefig('output/AQM_CorrelationMatrix.png')